# UFC PREDICTION MODEL 1.5 (Logistic Regression) ###

Notes:
- To start enviroment: venv\Scripts\Activate.ps1

### Import required libraries

In [33]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

# Model training
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.preprocessing import StandardScaler

# Model calibration (if needed later)
from sklearn.calibration import CalibratedClassifierCV

# Evaluation
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

### Import Data

In [34]:
df = pd.read_csv("../Data\large_dataset.csv")

# Reverse the dataset to begin with the oldest fight
df = df.iloc[::-1].reset_index(drop=True)


<>:1: SyntaxWarning: invalid escape sequence '\l'
<>:1: SyntaxWarning: invalid escape sequence '\l'
C:\Users\fancy\AppData\Local\Temp\ipykernel_28436\3763960240.py:1: SyntaxWarning: invalid escape sequence '\l'
  df = pd.read_csv("../Data\large_dataset.csv")


### Setup Variables

In [35]:
# Create binary winner label
df['winner_binary'] = df['winner'].map({'Red': 1, 'Blue': 0})

diff_features = [
    'SLpM_total_diff', 'SApM_total_diff', 'sig_str_acc_total_diff',
    'td_acc_total_diff', 'str_def_total_diff', 'td_def_total_diff',
    'sub_avg_diff', 'td_avg_diff', 'age_diff', 'height_diff', 'reach_diff', 'wins_total_diff', 'losses_total_diff'
]

# Drop rows with missing values in key columns
df = df.dropna(subset=diff_features + ['winner_binary'])

# Reverse for chronological ordering
df = df.iloc[::-1].reset_index(drop=True)

# Split dataset
split_idx = int(len(df) * (2 / 3))
train_df = df.iloc[:split_idx]
test_df = df.iloc[split_idx:]

# Define features and targets
X_train = train_df[diff_features]
y_train = train_df['winner_binary']
X_test = test_df[diff_features]
y_test = test_df['winner_binary']

# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)




### Train the Model

In [36]:
# Train a logistic regression model
calibrated = CalibratedClassifierCV(
    LogisticRegression(
        penalty='l2',
        C=0.1,
        solver='liblinear',
        max_iter=1000,
        class_weight='balanced',
        random_state=42
    ),
    method='sigmoid',
    cv=5
)

calibrated.fit(X_train_scaled, y_train)
probs = calibrated.predict_proba(X_test_scaled)

### Predict and Evaluate

In [37]:
# Predict + evaluate
y_pred = calibrated.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)

# Display results
print("Accuracy:", accuracy)
print("\nClassification Report:\n", report)
print("Confusion Matrix:\n", conf_matrix)

Accuracy: 0.6306898169873298

Classification Report:
               precision    recall  f1-score   support

           0       0.41      0.59      0.48       617
           1       0.80      0.65      0.71      1514

    accuracy                           0.63      2131
   macro avg       0.60      0.62      0.60      2131
weighted avg       0.68      0.63      0.65      2131

Confusion Matrix:
 [[365 252]
 [535 979]]


c:\CODE\Python\ML Projects\UFC-Prediction\venv\Lib\site-packages\sklearn\utils\validation.py:2742: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(
c:\CODE\Python\ML Projects\UFC-Prediction\venv\Lib\site-packages\sklearn\utils\validation.py:2742: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(
c:\CODE\Python\ML Projects\UFC-Prediction\venv\Lib\site-packages\sklearn\utils\validation.py:2742: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(
c:\CODE\Python\ML Projects\UFC-Prediction\venv\Lib\site-packages\sklearn\utils\validation.py:2742: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(
c:\CODE\Python\ML Projects\UFC-Prediction\venv\Lib\site-packages\sklearn\utils\validation.py:2742: UserWarning: X has feature names, but LogisticRegression was fitted without f

In [38]:
# Save model for external use
joblib.dump(calibrated, "LGReg_ufc_model.pkl")
joblib.dump(scaler, "LGReg_scaler.pkl")


['LGReg_scaler.pkl']

### Visualize/Analyze

In [39]:
# importances = model.feature_importances_

# importance_df = pd.DataFrame({
#     'Feature': diff_features,
#     'Importance': importances
# }).sort_values(by='Importance', ascending=False)


# plt.figure(figsize=(10, 6))
# plt.barh(importance_df['Feature'], importance_df['Importance'])
# plt.xlabel('Importance Score')
# plt.title('Feature Importance - UFC XGBoost Model')
# plt.gca().invert_yaxis()  # Most important at the top
# plt.tight_layout()
# plt.show()

In [40]:
# Example: Predict probabilities for the test set and display for first 5 fights
probs = calibrated.predict_proba(X_test)

# Find the index for class 1 (Red wins)
red_idx = list(calibrated.classes_).index(1)
blue_idx = list(calibrated.classes_).index(0)

for i in range(5):
    red_prob = probs[i, red_idx]
    blue_prob = probs[i, blue_idx]
    red_fighter = test_df.iloc[i]['R_fighter'] if 'R_fighter' in test_df.columns else 'Red Fighter'
    blue_fighter = test_df.iloc[i]['B_fighter'] if 'B_fighter' in test_df.columns else 'Blue Fighter'
    print(f"{red_fighter} wins probability: {red_prob:.3f}")
    print(f"{blue_fighter} wins probability: {blue_prob:.3f}\n")

c:\CODE\Python\ML Projects\UFC-Prediction\venv\Lib\site-packages\sklearn\utils\validation.py:2742: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(
c:\CODE\Python\ML Projects\UFC-Prediction\venv\Lib\site-packages\sklearn\utils\validation.py:2742: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(
c:\CODE\Python\ML Projects\UFC-Prediction\venv\Lib\site-packages\sklearn\utils\validation.py:2742: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(
c:\CODE\Python\ML Projects\UFC-Prediction\venv\Lib\site-packages\sklearn\utils\validation.py:2742: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(
c:\CODE\Python\ML Projects\UFC-Prediction\venv\Lib\site-packages\sklearn\utils\validation.py:2742: UserWarning: X has feature names, but LogisticRegression was fitted without f

Red Fighter wins probability: 0.042
Blue Fighter wins probability: 0.958

Red Fighter wins probability: 0.650
Blue Fighter wins probability: 0.350

Red Fighter wins probability: 0.988
Blue Fighter wins probability: 0.012

Red Fighter wins probability: 1.000
Blue Fighter wins probability: 0.000

Red Fighter wins probability: 1.000
Blue Fighter wins probability: 0.000

